# Khôi phục Phase 4 — checkpoint bị thiếu

Chạy trên **runtime Colab cũ** còn `/content/wdmamba_phase4/all/wavelite_quick_v2`.
Notebook tải một bộ checkpoint dùng chung, chạy lại các job lỗi bằng source và lịch train
đã lưu của lần chạy đó. Dataset và kết quả đã hoàn tất được giữ nguyên; I-HAZE đã xong
không train hoặc benchmark lại. Không cần đổi `RUN_NAME`.

Nếu Google Drive hết quota, đặt `CHECKPOINT_ZIP` tới archive local đã tải sẵn.
Kết quả tổng hợp sẽ nằm trong `results` của run cũ.

In [ ]:
RECOVERY_SOURCES = {'checkpoint_setup.py': '"""Shared checkpoint bootstrap. Does not modify model/training sources or artifacts."""\nfrom pathlib import Path\nimport json\nimport subprocess\nimport sys\nimport zipfile\n\n\nWEIGHT_NAMES = {\'o-hazy\': \'Ohaze_1200_1600_27.22.pth\',\n                \'reside6k\': \'reside6k_32.15.pth\',\n                \'sots-its\': \'reside6k_32.15.pth\', \'sots-ots\': \'reside6k_32.15.pth\'}\n\n\ndef download_archive(target):\n    try:\n        import gdown\n    except ImportError:\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'gdown\'], check=True)\n        import gdown\n    result = gdown.download(id=\'1Vm-1mtMSc9YIUICq4nWKu-4AHCtYnPpF\', output=str(target),\n                            quiet=False, use_cookies=False, resume=True)\n    if result is None:\n        raise RuntimeError(\'Google Drive không trả về ZIP trọng số. Kiểm tra quota hoặc đặt CHECKPOINT_ZIP local.\')\n\n\ndef find_weight(roots, filename):\n    for root in roots:\n        root = Path(root).expanduser()\n        if root.is_dir():\n            found = sorted(root.rglob(filename))\n            found = [p for p in found if p.is_file() and p.stat().st_size]\n            if found:\n                return found[0].resolve()\n    return None\n\n\ndef ensure_shared_weights(configs, cache_root, download=download_archive):\n    """Populate shared cache once; keep failures per dataset and leave explicit paths alone."""\n    cache = Path(cache_root).expanduser().resolve()\n    cache.mkdir(parents=True, exist_ok=True)\n    missing, errors = [], {}\n    for cfg in configs:\n        name = cfg[\'dataset_name\']\n        if name not in WEIGHT_NAMES:\n            continue  # I-HAZE never needs pretrained weights.\n        roots = cfg.setdefault(\'checkpoint_search_roots\', [])\n        if str(cache) not in roots:\n            roots.append(str(cache))\n        if cfg.get(\'repo_root\') and cfg[\'repo_root\'] not in roots:\n            roots.append(cfg[\'repo_root\'])\n        if cfg.get(\'checkpoint\') or (name != \'o-hazy\' and cfg.get(\'reside_checkpoint\')):\n            continue  # Explicit path validation remains in the runner.\n        search = [cfg.get(\'project_root\', \'\'), cfg.get(\'repo_root\', \'\'),\n                  Path(cfg[\'work_root\'])/\'weights\', *roots]\n        if not find_weight([p for p in search if p], WEIGHT_NAMES[name]):\n            missing.append(cfg)\n    if not missing:\n        return errors\n    # A supplied ZIP works even when network downloads are disabled.\n    provided = next((c[\'checkpoint_zip\'] for c in missing if c.get(\'checkpoint_zip\')), \'\')\n    downloadable = [c for c in missing if c.get(\'allow_weight_download\')]\n    if not provided and not downloadable:\n        return {c[\'dataset_name\']: \'Thiếu checkpoint và ALLOW_WEIGHT_DOWNLOAD=False.\' for c in missing}\n    archive = Path(provided).expanduser() if provided else cache/\'WDMamba_ckpts.zip\'\n    try:\n        if not provided and not zipfile.is_zipfile(archive):\n            partial = archive.with_suffix(\'.zip.partial\')\n            print(\'Đang tải bộ checkpoint dùng chung cho SOTS và O-HAZY...\', flush=True)\n            download(partial)\n            if not zipfile.is_zipfile(partial):\n                raise ValueError(\'File tải về không phải ZIP trọng số hợp lệ.\')\n            partial.replace(archive)\n        target = cache/\'unzipped\'; target.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(archive) as z:\n            if z.testzip() is not None:\n                raise ValueError(\'ZIP trọng số bị lỗi CRC.\')\n            for item in z.infolist():\n                if not (target/item.filename).resolve().is_relative_to(target):\n                    raise ValueError(\'ZIP chứa đường dẫn ngoài thư mục giải nén.\')\n            z.extractall(target)\n        for cfg in missing:\n            name = cfg[\'dataset_name\']\n            if not find_weight([cache], WEIGHT_NAMES[name]):\n                errors[name] = f\'ZIP không chứa {WEIGHT_NAMES[name]}.\'\n    except Exception as exc:\n        errors = {c[\'dataset_name\']: str(exc) for c in missing}\n    for name, message in errors.items():\n        print(f\'{name}: chuẩn bị checkpoint lỗi: {message}\', flush=True)\n    return errors\n\n\nif __name__ == \'__main__\':\n    path = Path(sys.argv[1])\n    bundle = json.loads(path.read_text(encoding=\'utf-8\'))\n    bundle[\'errors\'] = ensure_shared_weights(bundle[\'configs\'], bundle[\'cache_root\'])\n    path.write_text(json.dumps(bundle, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n', 'recover_phase4.py': '"""Retry failed jobs using their original saved sources; reuse completed results."""\nfrom pathlib import Path\nimport json\nimport subprocess\nimport sys\nfrom checkpoint_setup import ensure_shared_weights\n\n\nORDER = (\'sots-its\', \'sots-ots\', \'o-hazy\', \'i-haze\')\n\n\ndef recover_all(root, python, local_zip=\'\', invoke=None):\n    root = Path(root).expanduser().resolve()\n    configs = []\n    for name in ORDER:\n        path = root/\'datasets\'/name/\'config.json\'\n        if not path.is_file():\n            raise FileNotFoundError(f\'Không có {path}. Cần dùng runtime cũ còn thư mục run đã chạy.\')\n        cfg = json.loads(path.read_text(encoding=\'utf-8\'))\n        if cfg[\'dataset_name\'] != name:\n            raise ValueError(f\'Config dataset không khớp: {path}\')\n        configs.append(cfg)\n    output = root/\'results\'; output.mkdir(parents=True, exist_ok=True)\n    status_path = output/\'suite_status.json\'\n    old = json.loads(status_path.read_text(encoding=\'utf-8\')) if status_path.is_file() else []\n    completed = {s[\'dataset\'] for s in old if s[\'status\'] == \'completed\'}\n    states, pending = [], []\n    for cfg in configs:\n        dest = Path(cfg[\'output_root\'])\n        done = (cfg[\'dataset_name\'] in completed and (dest/\'summary.csv\').is_file()\n                and (dest/\'metadata.json\').is_file())\n        states.append({\'dataset\': cfg[\'dataset_name\'], \'status\': \'completed\' if done else \'ready\', \'error\': \'\'})\n        if done:\n            print(f\'{cfg["dataset_name"]}: giữ kết quả đã hoàn tất, không train/benchmark lại.\', flush=True)\n        else:\n            cfg[\'allow_weight_download\'] = True\n            if local_zip:\n                cfg[\'checkpoint_zip\'] = local_zip\n            pending.append(cfg)\n    errors = ensure_shared_weights(pending, root/\'checkpoints\')\n\n    def run(cfg, action):\n        args = [str(python), \'-u\', str(Path(cfg[\'work_root\'])/\'phase4_runner.py\'),\n                str(Path(cfg[\'work_root\'])/\'config.json\'), action]\n        print(\'$\', \' \'.join(args), flush=True)\n        with (root/f\'recover_{cfg["dataset_name"]}_{action}.log\').open(\'a\', encoding=\'utf-8\') as log:\n            with subprocess.Popen(args, cwd=cfg[\'repo_root\'], stdout=subprocess.PIPE,\n                                  stderr=subprocess.STDOUT, text=True, encoding=\'utf-8\', errors=\'replace\') as proc:\n                try:\n                    for line in proc.stdout:\n                        print(line, end=\'\', flush=True); log.write(line); log.flush()\n                    code = proc.wait()\n                except BaseException:\n                    proc.kill(); proc.wait(); raise\n            if code:\n                raise subprocess.CalledProcessError(code, args)\n\n    invoke = invoke or run\n    batch_ns = {\'__name__\': \'phase4_batch_recovery\'}\n    batch_path = Path(configs[0][\'work_root\'])/\'phase4_batch.py\'\n    exec(compile(batch_path.read_text(encoding=\'utf-8-sig\'), str(batch_path), \'exec\'), batch_ns)\n    for cfg, state in zip(configs, states):\n        if state[\'status\'] == \'completed\':\n            continue\n        state[\'status\'] = \'prepare_failed\'\n        try:\n            if cfg[\'dataset_name\'] in errors:\n                raise RuntimeError(errors[cfg[\'dataset_name\']])\n            # Download/search options are not part of the training identity.\n            # Keep original runner/architecture sources and schedules untouched.\n            (Path(cfg[\'work_root\'])/\'config.json\').write_text(\n                json.dumps(cfg, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n            invoke(cfg, \'prepare\')\n            state[\'status\'] = \'run_failed\'\n            invoke(cfg, \'auto\')\n            state[\'status\'] = \'completed\'\n        except Exception as exc:\n            state[\'error\'] = str(exc)\n            print(f\'{cfg["dataset_name"]}: {exc}\', flush=True)\n        finally:\n            batch_ns[\'save_status\'](output, states)\n            batch_ns[\'collect_reports\'](configs, states, output)\n    missing = [s[\'dataset\'] for s in states if s[\'status\'] != \'completed\']\n    if missing:\n        raise RuntimeError(\'Lượt all chưa hoàn tất: \' + \', \'.join(missing) + \'. Xem suite_status.json và log khôi phục.\')\n    print(\'Hoàn tất đủ 4/4 dataset:\', output/\'final_report_all.md\', flush=True)\n    return states\n\n\nif __name__ == \'__main__\':\n    recover_all(sys.argv[1], sys.argv[2], sys.argv[3] if len(sys.argv) > 3 else \'\')\n'}

from pathlib import Path
import subprocess

RUN_ROOT = Path('/content/wdmamba_phase4/all/wavelite_quick_v2')
PYTHON = Path('/content/wdmamba_phase4_env311/bin/python')
CHECKPOINT_ZIP = ''  # tùy chọn: /content/drive/MyDrive/WDMamba_ckpts.zip

if not (RUN_ROOT/'datasets').is_dir():
    raise FileNotFoundError('Không còn run cũ. Dùng runtime cũ hoặc khôi phục đầy đủ thư mục kết quả.')
if not PYTHON.is_file():
    raise FileNotFoundError('Không có Python GPU của run cũ; sửa PYTHON tới môi trường đã dùng.')
tools_root = RUN_ROOT/'recovery_tools'; tools_root.mkdir(exist_ok=True)
for name, source in RECOVERY_SOURCES.items():
    (tools_root/name).write_text(source, encoding='utf-8')
args = [str(PYTHON), '-u', str(tools_root/'recover_phase4.py'), str(RUN_ROOT), str(PYTHON), CHECKPOINT_ZIP]
with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding='utf-8', errors='replace') as proc:
    try:
        for line in proc.stdout: print(line, end='', flush=True)
        rc = proc.wait()
    except BaseException:
        proc.kill(); proc.wait(); raise
if rc: raise subprocess.CalledProcessError(rc, args)


In [ ]:

from IPython.display import Markdown, display
import shutil
display(Markdown((RUN_ROOT/'results/final_report_all.md').read_text(encoding='utf-8')))
archive = shutil.make_archive(str(RUN_ROOT.parent/(RUN_ROOT.name+'_recovered')), 'zip', RUN_ROOT)
print('Artifact:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass
